In [0]:
CATALOG = "ecommerce_catalog"
SCHEMA = "ecommerce_schema"

BRONZE_PATH = "/Volumes/ecommerce_catalog/ecommerce_schema/bronze"

CUSTOMERS_BRONZE = f"{BRONZE_PATH}/bronze_customers"
ORDERS_BRONZE = f"{BRONZE_PATH}/bronze_orders"



In [0]:
customers = spark.read.format("delta").load(CUSTOMERS_BRONZE)

orders = spark.read.format("delta").load(ORDERS_BRONZE)

print("Customer rows:", customers.count())
print("Customer columns:", len(customers.columns))
customers.printSchema()


print("Order rows:", orders.count())
print("Order columns:", len(orders.columns))
orders.printSchema()

In [0]:
from pyspark.sql.functions import col, count

customer_duplicates = customers \
    .groupBy("customer_id") \
    .agg(count("*").alias("count")) \
    .filter(col("count") > 1)

customer_duplicates.display()


order_duplicates = orders \
    .groupBy("order_id") \
    .agg(count("*").alias("count")) \
    .filter(col("count") > 1)

order_duplicates.display()

In [0]:
customers = spark.read.format("delta").load(CUSTOMERS_BRONZE)

orders = spark.read.format("delta").load(ORDERS_BRONZE)

customers.filter(
    col("customer_id").isNull() |
    col("customer_name").isNull() |
    col("email").isNull() |
    col("city").isNull() |
    col("customer_type").isNull()
).display()


orders.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("order_date").isNull() |
    col("city").isNull() |
    col("product_category").isNull() |
    col("quantity").isNull() |
    col("unit_price").isNull() |
    col("status").isNull() |
    col("payment_method").isNull()
).display()

In [0]:
customers = spark.read.format("delta").load(CUSTOMERS_BRONZE)

orders = spark.read.format("delta").load(ORDERS_BRONZE)


customers.select("customer_type") \
    .distinct() \
    .display()


customers.select("city") \
    .distinct() \
    .display()

orders.select("status") \
    .distinct() \
    .display()

orders.select("payment_method") \
    .distinct() \
    .display()


orders.select("product_category") \
    .distinct() \
    .display()

In [0]:
customers = spark.read.format("delta").load(CUSTOMERS_BRONZE)

orders = spark.read.format("delta").load(ORDERS_BRONZE)

orders.filter(
    col("quantity") <= 0
).display()


orders.filter(
    col("unit_price") < 0
).display()


orders.filter(
    col("unit_price").isNull()
).display()

In [0]:
customers = spark.read.format("delta").load(CUSTOMERS_BRONZE)

orders = spark.read.format("delta").load(ORDERS_BRONZE)


unmatched_customers = orders.alias("o") \
    .join(
        customers.alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left"
    ) \
    .filter(col("c.customer_id").isNull())

unmatched_customers.display()

print("Unmatched customer references:", unmatched_customers.count())

In [0]:
orders = spark.read.format("delta").load(ORDERS_BRONZE)

orders.select("order_date") \
    .distinct() \
    .display()

In [0]:
from pyspark.sql.functions import length, trim
customers = spark.read.format("delta").load(CUSTOMERS_BRONZE)

orders = spark.read.format("delta").load(ORDERS_BRONZE)

orders.select(
    "city",
    length(col("city")).alias("length"),
    length(trim(col("city"))).alias("trimmed_length")
).filter(
    col("length") != col("trimmed_length")
).display()

CUSTOMER CLEANING

In [0]:
# Save valid customers as CSV
VALID_FOLDER = "/Volumes/ecommerce_catalog/ecommerce_schema/silver/clean_customers"
VALID_FILE = f"{VALID_FOLDER}/clean_customers.csv"

dbutils.fs.rm(VALID_FOLDER, True)

valid_customers.drop("is_valid") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(VALID_FOLDER + "/temp")

for file in dbutils.fs.ls(VALID_FOLDER + "/temp"):
    if file.name.endswith(".csv"):
        dbutils.fs.cp(
            file.path,
            VALID_FILE
        )

dbutils.fs.rm(VALID_FOLDER + "/temp", True)

print(f"✓ Saved {valid_customers.count()} valid customers to {VALID_FILE}")


# Save invalid customers as CSV
INVALID_FOLDER = "/Volumes/ecommerce_catalog/ecommerce_schema/silver/invalid_customers"
INVALID_FILE = f"{INVALID_FOLDER}/invalid_customers.csv"

dbutils.fs.rm(INVALID_FOLDER, True)

invalid_customers.drop("is_valid") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(INVALID_FOLDER + "/temp")

for file in dbutils.fs.ls(INVALID_FOLDER + "/temp"):
    if file.name.endswith(".csv"):
        dbutils.fs.cp(
            file.path,
            INVALID_FILE
        )

dbutils.fs.rm(INVALID_FOLDER + "/temp", True)

print(f"✓ Saved {invalid_customers.count()} invalid customers to {INVALID_FILE}")

CLEAN ORDERS

In [0]:
# Load and validate orders
from pyspark.sql.functions import col, trim, lower, when

orders = spark.read.format("delta").load(ORDERS_BRONZE)

# Basic cleaning
orders_clean = orders \
    .withColumn("city", trim(lower(col("city")))) \
    .withColumn("status", trim(lower(col("status")))) \
    .withColumn("payment_method", trim(lower(col("payment_method")))) \
    .withColumn("product_category", trim(lower(col("product_category"))))

# Validate orders
orders_with_validation = orders_clean.withColumn(
    "is_valid",
    col("order_id").isNotNull() &
    col("customer_id").isNotNull() &
    col("order_date").isNotNull() &
    col("city").isNotNull() &
    col("product_category").isNotNull() &
    col("quantity").isNotNull() &
    (col("quantity") > 0) &
    col("unit_price").isNotNull() &
    (col("unit_price") >= 0) &
    col("status").isNotNull() &
    col("payment_method").isNotNull()
)

valid_orders = orders_with_validation.filter(col("is_valid") == True)
invalid_orders = orders_with_validation.filter(col("is_valid") == False)

# Save valid orders as CSV
VALID_FOLDER = "/Volumes/ecommerce_catalog/ecommerce_schema/silver/clean_orders"
VALID_FILE = f"{VALID_FOLDER}/clean_orders.csv"

dbutils.fs.rm(VALID_FOLDER, True)

valid_orders.drop("is_valid") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(VALID_FOLDER + "/temp")

for file in dbutils.fs.ls(VALID_FOLDER + "/temp"):
    if file.name.endswith(".csv"):
        dbutils.fs.cp(
            file.path,
            VALID_FILE
        )

dbutils.fs.rm(VALID_FOLDER + "/temp", True)

print(f"✓ Saved {valid_orders.count()} valid orders to {VALID_FILE}")


# Save invalid orders as CSV
INVALID_FOLDER = "/Volumes/ecommerce_catalog/ecommerce_schema/silver/invalid_orders"
INVALID_FILE = f"{INVALID_FOLDER}/invalid_orders.csv"

dbutils.fs.rm(INVALID_FOLDER, True)

invalid_orders.drop("is_valid") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(INVALID_FOLDER + "/temp")

for file in dbutils.fs.ls(INVALID_FOLDER + "/temp"):
    if file.name.endswith(".csv"):
        dbutils.fs.cp(
            file.path,
            INVALID_FILE
        )

dbutils.fs.rm(INVALID_FOLDER + "/temp", True)

print(f"✓ Saved {invalid_orders.count()} invalid orders to {INVALID_FILE}")